![](../resources/Custom_croptype_map.png)

### How to run this notebook?

You can use a preconfigured environment on VITO's [**Terrascope**](https://terrascope.be/en) platform to run this notebook.

Once you have a Terrascope account, simply click on the button below to get started.

<div class="alert alert-block alert-warning">When you click the button, you will be prompted with "Server Options".<br>
Make sure to select the <b>"Worldcereal"</b> image here. Did you choose "Terrascope" by accident?<br>
Then go to File > Hub Control Panel > Stop my server, and click the link below once again.</div>

<a href="https://notebooks.terrascope.be/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2FWorldCereal%2Fworldcereal-classification&urlpath=lab%2Ftree%2Fworldcereal-classification%2Fnotebooks%2Ftrainings%2F20260416_ESTELLA_workshop_Ghana.ipynb&branch=main"><img src="https://img.shields.io/badge/Run%20notebook%20on-Terrascope-brightgreen" alt="Run notebook" valign="middle"></a>


<div class="alert alert-block alert-warning">
<b>WARNING:</b> <br>
Every time you click the above link, the latest version of the notebook will be fetched, potentially leading to conflicts with changes you have made yourself.<br>
To avoid such code conflicts, we recommend you to <b>make a copy of the notebook and make changes only in your copied version</b>.
</div>

### Introduction

This notebook guides you through the process of training a custom crop type classification model for your area, season and crop types of interest and then apply it to a region of interest to generate a crop type map.

For training the model, you can use a combination of:
- publicly available reference data harmonized by the WorldCereal consortium;
- your own private reference data.

**We will go through the following steps:**

1) Upload a private dataset to the [WorldCereal reference data module (RDM)](https://rdm.esa-worldcereal.org/);
2) Extract the required EO and ancillary data matching your private dataset;
3) Train a classification model
4) Apply your model to a specific test area
5) Upscale your model to produce a map for a larger region

### Before you start

In order to run WorldCereal crop mapping jobs from this notebook, you need to create an account on the [Copernicus Data Space Ecosystem](https://dataspace.copernicus.eu/).<br>
This is free of charge and will grant you a number of free openEO processing credits to continue this demo.

#### Ensure proper access to functionality

Execute the next block of code to ensure this notebook has access to all functionalities needed in this exercise.

In [ ]:
# add parent dirctory to sys.path
import sys
sys.path.append('..')

### 1. Upload a private dataset

We have prepared a small dataset over Ghana which we will be treating as a "private dataset" in this exercise.

Execute the next cell to start the download of this dataset --> will be stored in folder `download`.

In [ ]:
from pathlib import Path
import geopandas as gpd

# Set target path for the data
target_path = Path.cwd() / "download/Ghana_test_data_2025.gpkg"
target_path.parent.mkdir(exist_ok=True)

# Download and extract the data if not already present
if not target_path.exists():
    url = "https://artifactory.vgt.vito.be/artifactory/auxdata-public/worldcereal/demo/Ghana/Ghana_test_data_2025.gpkg"
    # Download the file
    import urllib.request
    urllib.request.urlretrieve(url, target_path)
print(f"Data downloaded to: {target_path}")

# read file and show contents
gdf = gpd.read_file(target_path)
gdf.head()

Now that you have the test dataset, browse to the WorldCereal [RDM](https://rdm.esa-worldcereal.org) and:

- log in with your CDSE credentials
- hit the `Contribute` button on the home page to start the upload procedure

### 2. Run EO data extractions for your private dataset

Now navigate to the `worldcereal_private_extractions.ipynb` notebook located under worldcereal-classification > notebooks and run through the procedure to extract EO and ancillary data for your private dataset.

Alternatively, the notebook can be found using [this link](https://github.com/WorldCereal/worldcereal-classification/blob/main/notebooks/worldcereal_private_extractions.ipynb).

### 3. Train your model

Launch the application in the next cell to train your crop type model for Ghana!

Make sure to choose the option "**Full workflow**" in the welcome screen of the application.<br>

Then complete steps 1 --> 6 (stop before deploying your model in step 7).<br>

Brief outline of steps:
1. Retrieve existing reference data (make sure to select Ghana as AOI!)
2. Inspect the reference data
3. Choose your season of interest and align your reference data to that season
4. Compute Presto embeddings on your reference data
5. Select the classes you want to include in your model
6. Train your model

Once finalized, **continue with the next cell in this notebook**.

Later, you can still run steps 7 -> 9 (large scale map generation).

In [ ]:
from notebook_utils.classification_app import WorldCerealClassificationApp

app = WorldCerealClassificationApp().run()

### 4. Local Inference

As an alternative to the model deployment and inference steps as exposed in the application above (steps 7 -> 9), we offer here the possibility to quickly run your newly trained model on a test patch in Northern Ghana. For this patch, we have already fetched the required pre-processed time series inputs from CDSE using openEO workflows. For more information on this procedure, you can consult the [worldcereal_preprocessed_inputs notebook](https://github.com/WorldCereal/worldcereal-classification/blob/main/notebooks/worldcereal_preprocessed_inputs.ipynb).

The next cells load this prepared patch, show a random NDVI slice for context, and call `run_seasonal_inference()` with the seasonal backbone plus your custom croptype head to produce a croptype raster for the chosen season.

This step is meant for fast QA and tuning of inference knobs and post-processing; it does not replace the full openEO production run. The output is written as a GeoTIFF under `./local_inference/` with a filename that includes the patch and head package, so you can inspect it in your GIS tools or compare with reference data.


**Download and visualize one patch**

(no need to change anything in the next cell, just execute)

In [ ]:
import random
import numpy as np
from pathlib import Path
import urllib.request

from pyproj import CRS
import xarray as xr

from notebook_utils.preprocessed_inputs import get_band_statistics_netcdf, visualize_timeseries_netcdf

# Download the demo patch from artifactory if the requested patch is missing locally.
local_file_path = Path(f"./local_inference/input_patches/preprocessed-inputs_2024-11-01_2025-10-31_1_30PYR_0.nc")
if not local_file_path.exists():
    local_file_path.parent.mkdir(parents=True, exist_ok=True)
    if not local_file_path.exists():
        print(f"Downloading demo preprocessed inputs to {local_file_path}...")
        remote_url = (
            f"https://artifactory.vgt.vito.be/artifactory/auxdata-public/worldcereal/demo/Ghana/preprocessed-inputs_2024-11-01_2025-10-31_1_30PYR_0.nc"
        )
        urllib.request.urlretrieve(remote_url, local_file_path)

ds = xr.open_dataset(local_file_path, engine="netcdf4")
epsg = CRS.from_wkt(ds.crs.attrs["spatial_ref"]).to_epsg()

# Visualize a random timestamp from the patch
ndvi = (ds["S2-L2A-B08"] - ds["S2-L2A-B04"]) / (ds["S2-L2A-B08"] + ds["S2-L2A-B04"])
timestamp_ind = np.random.randint(0, ndvi.shape[0])
ndvi.isel(t=timestamp_ind).plot(cmap="RdYlGn", vmin=-0.8, vmax=0.8)

# Show band statistics
stats = get_band_statistics_netcdf(ds)

# Visualize random time series
visualize_timeseries_netcdf(ds, band="NDVI", npixels=50, random_seed=42)

**Set processing parameters**

Configure the inference and post-processing parameters:

- `mask_cropland`: Apply cropland masking during croptype predictions
- `enable_cropland_head`: Also export cropland probability rasters for quality assurance
- `export_class_probs`: Emit per-class probability layers for each crop type
- `croptype_postprocess_enabled`: Apply spatial post-processing to croptype classifications
- `croptype_postprocess_method`: Post-processing algorithm (e.g., "majority_vote")
- `croptype_postprocess_kernel`: Kernel size for post-processing filter
- `cropland_postprocess_enabled`: Apply spatial post-processing to cropland mask
- `cropland_postprocess_method`: Post-processing algorithm for cropland
- `cropland_postprocess_kernel`: Kernel size for cropland post-processing

(just execute the next cell if you want to accept the default settings)

In [ ]:
mask_cropland = True  # cropland masking using default cropland model
enable_cropland_head = True  # also export cropland rasters for QA
enable_croptype_head = True  # whether to run the croptype head at all (if False, only cropland predictions will be made)
export_class_probs = True  # export per-class probabilities in the crop type product

croptype_postprocess_enabled = True
croptype_postprocess_method = "majority_vote"
croptype_postprocess_kernel = 3

cropland_postprocess_enabled = True
cropland_postprocess_method = "majority_vote"
cropland_postprocess_kernel = 3

**Specify year and season**

For your information, we first display for which season_id and growing season period you have trained your model:

In [ ]:
season_id = app.season_id
print(f"Model trained for season id: {season_id}")
window_dt = app.season_window.to_datetime()
hint_start = window_dt[0].strftime("%d %b")
hint_end = window_dt[1].strftime("%d %b")
print(f"Model trained for season: {hint_start} - {hint_end}")

In the next cell, you can now manually set a start and end date for your season using the slider.

**NOTE**: for the preprocessed input patches you are working with here, we only have data available between 2024-11-01 and 2025-10-30 !!!

In [ ]:
from datetime import datetime
from notebook_utils.dateslider import date_slider

start_available_data = datetime(2024, 11, 1)
end_available_data = datetime(2025, 10, 30)
slider = date_slider(year_selector=False, start_date=start_available_data, end_date=end_available_data)

**Run local inference**

(no need to change anything below)

In [ ]:
import json
from worldcereal.openeo.parameters import DEFAULT_SEASONAL_MODEL_URL

from notebook_utils.local_inference import (
    run_seasonal_inference,
    build_postprocess_spec,
    classification_to_geotiff,
)

# Get season from slider
selected = slider.get_selection()
season_window = selected.season_window
start_season = str(season_window.start_date)
end_season = str(season_window.end_date)
season_windows = {
    season_id: (start_season, end_season)
}

# Post-processing parameters
croptype_postprocess = build_postprocess_spec(
    enabled=croptype_postprocess_enabled,
    method=croptype_postprocess_method,
    kernel_size=croptype_postprocess_kernel,
)
cropland_postprocess = build_postprocess_spec(
    enabled=cropland_postprocess_enabled,
    method=cropland_postprocess_method,
    kernel_size=cropland_postprocess_kernel,
)

# Get trained model paths from the app
seasonal_model_zip_path = DEFAULT_SEASONAL_MODEL_URL

# Run inference
classification_result = run_seasonal_inference(
    ds,  # or local_file_path
    seasonal_model_zip=seasonal_model_zip_path,
    croptype_head_zip=app.head_package_path,  # the crop type model you trained in the app
    season_ids=[season_id],
    season_windows=season_windows,
    enforce_cropland_gate=mask_cropland,
    export_class_probabilities=export_class_probs,
    enable_cropland_head=enable_cropland_head,
    enable_croptype_head=enable_croptype_head,
    croptype_postprocess=croptype_postprocess,
    cropland_postprocess=cropland_postprocess,
    as_dataset=False,  # DataArray for GeoTIFF
)

# Specify output directory and name for the classification result
output_dir = Path("./local_inference")
output_dir.mkdir(parents=True, exist_ok=True)
# retrieve model name
head_tag = Path(app.head_package_path).stem
classification_result_path = output_dir / f"croptype_{season_id}_{head_tag}.tif"

# get model class names from the model config file
head_output_dir = app.head_output_path
head_config_path = head_output_dir / "config.json"
if not head_config_path.exists():
    raise FileNotFoundError(
        f"Torch head config not found at {head_config_path}. Check the training logs above."
    )
with head_config_path.open() as fp:
    head_config = json.load(fp)
class_map = {i: name for i, name in enumerate(head_config["classes_list"])}

# Finally, save to geotiff
classification_to_geotiff(classification_result, epsg, classification_result_path, class_map)

print(f"Classification result saved to {classification_result_path}")

The following cell helps you to quickly visualize your maps.

For more detailed inspection (for instance of probabilities), we advise to load the product(s) in QGIS.

In [ ]:
from notebook_utils.local_inference import isolate_cropland_croptype_products
from notebook_utils.visualization import visualize_products

# split one raster into separate cropland and crop type products
products = isolate_cropland_croptype_products(
    classification_result_path,
    enable_cropland_head)

# Get class names from your model and convert to look-up table compatible with visualization function
lut_croptype = {v: k for k, v in sorted(class_map.items(), key=lambda kv: kv[0])}
luts = {'croptype': lut_croptype}

# Run simple visualization of the classification bands
visualize_products(products, luts=luts)

Congratulations, you have reached the end of this demo!

You can now go back to the app and run through steps 7 - 9 to launch a map production job in your area of choice!